# Notebook 07: Verify & Audit Dataset V2 & Huấn Luyện Thực Nghiệm Trên Google Colab

> **Mục tiêu của Notebook:**
> 1. **Giai đoạn 1 (Audit & Verification):** Quét đệ quy tìm các file zip dataset-v2, kiểm tra checksum SHA256, phân tích cấu trúc đóng gói `KLTN_prep_v2` (kho ảnh `images/aug`, `images/train`, `images/test`), kiểm tra tính toàn vẹn qua danh sách `train_*.txt`, `val_*.txt`, `test_*.txt`, lọc bỏ file text/comment `#`, kiểm tra tọa độ bbox và tự động sinh file `data.yaml` chuẩn cho YOLO11.
> 2. **Giai đoạn 2 (Training & Evaluation):** Cho phép chọn nhanh tỉ lệ split (ví dụ: `70_15_15`), giải nén siêu tốc lên SSD Colab và chạy thực nghiệm Baseline + CBAM (hỗ trợ lưu checkpoint lên Drive và tự khôi phục khi bị ngắt kết nối/crash).

---

## 📌 Bước 1️⃣: Kết Nối Google Drive & Chuẩn Bị Môi Trường
> **Yêu cầu trước khi chạy:**
> - Tạo lối tắt (Shortcut) của thư mục chứa dataset hoặc thư mục chung `KLTN_38` / `KLTN-2026-testingNtraining` vào **My Drive** của bạn.

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Tự động nhận diện thư mục dataset-v2 trên Drive
POSSIBLE_DATASET_PATHS = [
    "/content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/dataset-v2",
    "/content/drive/MyDrive/KLTN_38/dataset/dataset-v2",
    "/content/drive/MyDrive/dataset/dataset-v2",
    "/content/drive/MyDrive/dataset-v2"
]

DATASET_V2_DIR = None
for p in POSSIBLE_DATASET_PATHS:
    if os.path.exists(p):
        DATASET_V2_DIR = p
        break

if DATASET_V2_DIR is None:
    my_drive = Path("/content/drive/MyDrive")
    found = list(my_drive.rglob("dataset-v2"))
    if found:
        DATASET_V2_DIR = str(found[0])

if DATASET_V2_DIR is None:
    print("⚠️ Chưa tự động tìm thấy thư mục dataset-v2 theo đường dẫn mặc định.")
    print("👉 Hãy gán thủ công DATASET_V2_DIR nếu bạn đặt tên khác.")
    DATASET_V2_DIR = "/content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/dataset-v2"
else:
    print(f"✅ Đã tìm thấy thư mục Dataset V2 tại: {DATASET_V2_DIR}")

# Clone repo mã nguồn
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git /content/klcn2026
%cd /content/klcn2026
!git checkout main
!git pull origin main

# Cài đặt thư viện
!pip install -q ultralytics albumentations tabulate pyyaml
print("\n🚀 Môi trường đã sẵn sàng!")

## 🔍 Bước 2️⃣: GIAI ĐOẠN 1 — Kiểm Tra Toàn Vẹn SHA256 & Phân Tích Cấu Trúc File ZIP
> Cell này quét 3 file zip (`80_10_10`, `70_15_15`, `60_20_20`) để:
> 1. Tính toán mã băm SHA256 và đối chiếu với file `.sha256`.
> 2. Đọc trực tiếp các file danh sách `train_*.txt`, `val_*.txt`, `test_*.txt` bên trong gói `KLTN_prep_v2` để tính tỉ lệ phân chia thực tế chính xác 100%.

In [ ]:
import hashlib
import zipfile
import os
from pathlib import Path
from tabulate import tabulate

assert DATASET_V2_DIR and os.path.exists(DATASET_V2_DIR), f"❌ Thư mục không tồn tại: {DATASET_V2_DIR}"
BASE = Path(DATASET_V2_DIR)

SPLITS = {"80_10_10": (80, 10, 10), "70_15_15": (70, 15, 15), "60_20_20": (60, 20, 20)}

def find_zip(key):
    hits = list(BASE.rglob(f"KLTN_prep_v2_{key}.zip"))
    return hits[0] if hits else None

def sha256(p, chunk=1<<24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

rows = []
found_zips = {}

print("=" * 80)
print("🔍 BẮT ĐẦU KIỂM TRA SHA256 VÀ PHÂN TÍCH TỆP ZIP TRÊN GOOGLE DRIVE")
print("=" * 80)

for key, ratio in SPLITS.items():
    zp = find_zip(key)
    if zp is None:
        rows.append([key, "MISSING", "N/A", "N/A", "❌ Không tìm thấy zip"])
        continue

    found_zips[key] = str(zp)
    shap = zp.with_name(zp.name + ".sha256")
    exp = shap.read_text(encoding="utf-8").split()[0].lower() if shap.exists() else None
    act = sha256(zp).lower()
    sha_st = "⚠️ NO .sha256" if exp is None else ("✅ KHỚP" if act == exp else "❌ LỆCH")

    try:
        with zipfile.ZipFile(zp) as z:
            namelist = [n for n in z.namelist() if not n.endswith("/")]

            # Tìm các file txt split: train_*.txt, val_*.txt, test_*.txt
            def read_split_lines(split_name):
                target = [n for n in namelist if f"{split_name}_{key}.txt" in n or (f"{split_name}.txt" in n and "split" in n)]
                if not target:
                    target = [n for n in namelist if n.endswith(f"{split_name}_{key}.txt")]
                if target:
                    content = z.read(target[0]).decode('utf-8', errors='ignore')
                    lines = [line.strip() for line in content.splitlines() if line.strip() and not line.strip().startswith("#")]
                    return set(lines)
                return set()

            train_lines = read_split_lines("train")
            val_lines = read_split_lines("val")
            test_lines = read_split_lines("test")

            n_train, n_val, n_test = len(train_lines), len(val_lines), len(test_lines)
            tot = n_train + n_val + n_test

            if tot > 0:
                actual = (round(100 * n_train / tot, 1), round(100 * n_val / tot, 1), round(100 * n_test / tot, 1))
                ratio_ok = all(abs(a - e) < 1.5 for a, e in zip(actual, ratio))
                split_summary = f"Train: {n_train} | Val: {n_val} | Test: {n_test} → {actual}% {'✅' if ratio_ok else '⚠️'}"
                
                # Kiểm tra leakage giữa các tập
                stems = lambda S: {Path(x).stem for x in S}
                overlap = len(stems(train_lines) & stems(val_lines)) + len(stems(train_lines) & stems(test_lines)) + len(stems(val_lines) & stems(test_lines))
                leakage_str = f"Trùng split: {overlap} {'✅' if overlap == 0 else '❌'}"
            else:
                img_count = sum(1 for n in namelist if n.lower().endswith(('.jpg', '.png', '.jpeg')))
                split_summary = f"Tổng {img_count} ảnh (Chưa đọc được file split list)"
                leakage_str = "-"

            rows.append([key, zp.name, f"{sha_st} ({act[:8]})", split_summary, leakage_str])
    except Exception as e:
        rows.append([key, zp.name, "CORRUPT", "Error", str(e)])

print(tabulate(rows, headers=["Tỉ lệ", "Zip", "SHA256", "Số ảnh & Tỉ lệ phân chia thực tế", "Leakage Check"], tablefmt="grid"))

## 🧪 Bước 3️⃣: Giải Nén Thử Nghiệm & Audit Chi Tiết Nhãn
> Chọn 1 bộ dataset (mặc định: `70_15_15`) để giải nén lên ổ cứng SSD cục bộ của Colab (`/content/dataset_v2/`).
> Sau đó chạy script kiểm tra nhãn:
> - Tự động bỏ qua các file cấu hình, file mô tả hoặc dòng comment `#`.
> - Kiểm tra Class ID có hợp lệ trong khoảng `0..47`.
> - Kiểm tra tọa độ Bounding Box `[x_center, y_center, width, height]` có chuẩn hóa đúng `[0, 1]`.

In [ ]:
import os
import yaml
from glob import glob
from pathlib import Path

SELECTED_RATIO = "70_15_15"  # Tùy chọn: '80_10_10', '70_15_15', '60_20_20'

TARGET_ZIP = found_zips.get(SELECTED_RATIO) if 'found_zips' in locals() and SELECTED_RATIO in found_zips else None
if TARGET_ZIP is None:
    hits = list(Path(DATASET_V2_DIR).rglob(f"KLTN_prep_v2_{SELECTED_RATIO}.zip"))
    assert hits, f"❌ Không tìm thấy zip cho {SELECTED_RATIO}"
    TARGET_ZIP = str(hits[0])

EXTRACT_DIR = f"/content/dataset_v2_{SELECTED_RATIO}"

print(f"📦 Đang giải nén {TARGET_ZIP} vào {EXTRACT_DIR}...")
!mkdir -p "{EXTRACT_DIR}"
!unzip -q "{TARGET_ZIP}" -d "{EXTRACT_DIR}"
print(f"✅ Giải nén hoàn tất vào {EXTRACT_DIR}!")

print("\n📂 Cấu trúc thư mục mức 1 và 2:")
!ls -la "{EXTRACT_DIR}"
if os.path.exists(f"{EXTRACT_DIR}/KLTN_prep_v2"):
    !ls -la "{EXTRACT_DIR}/KLTN_prep_v2"

# Tìm đúng các file nhãn YOLO trong thư mục labels/
all_label_files = glob(f"{EXTRACT_DIR}/**/labels/**/*.txt", recursive=True)
if not all_label_files:
    # Fallback nếu labels đặt phẳng
    candidates = glob(f"{EXTRACT_DIR}/**/*.txt", recursive=True)
    all_label_files = [f for f in candidates if not any(x in f for x in ["train_", "val_", "test_", "requirements", "config", "classes", ".sha256"])]

print(f"\n🔍 Tìm thấy tổng cộng {len(all_label_files)} file nhãn khuyết tật (.txt).")

invalid_classes = set()
invalid_bboxes = 0
total_objects = 0

for lf in all_label_files:
    with open(lf, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            clean_line = line.strip()
            # Bỏ qua dòng trống hoặc dòng comment
            if not clean_line or clean_line.startswith("#"):
                continue
            parts = clean_line.split()
            if len(parts) >= 5:
                # Thử parse class ID, bỏ qua nếu dòng header/text
                try:
                    cls_id = int(parts[0])
                except ValueError:
                    continue
                total_objects += 1
                if cls_id < 0 or cls_id >= 48:
                    invalid_classes.add(cls_id)
                coords = [float(x) for x in parts[1:5]]
                if any(c < 0.0 or c > 1.0 for c in coords):
                    invalid_bboxes += 1

print(f"📊 Tổng số bounding boxes: {total_objects}")
if invalid_classes:
    print(f"❌ CẢNH BÁO: Phát hiện Class ID nằm ngoài dải 0..47: {invalid_classes}")
else:
    print("✅ Toàn bộ Class ID đều hợp lệ trong khoảng 0..47!")

if invalid_bboxes > 0:
    print(f"⚠️ Có {invalid_bboxes} bbox có tọa độ vượt ngưỡng [0, 1].")
else:
    print("✅ Toàn bộ tọa độ Bounding Box đều chuẩn hóa đúng trong [0, 1]!")

## 🛠️ Bước 4️⃣: Tự Động Tạo / Chuẩn Hóa File `data.yaml` Tuyệt Đối
> Tự động tìm thư mục `KLTN_prep_v2` và kiểm tra file YAML có sẵn (`data_70_15_15.yaml`...) hoặc file `.txt` list để trỏ đường dẫn tuyệt đối cho YOLO huấn luyện.

In [ ]:
import yaml
from pathlib import Path

# Tìm thư mục gốc của bộ dữ liệu sau khi giải nén (có thể nằm trong KLTN_prep_v2/)
extract_p = Path(EXTRACT_DIR)
if (extract_p / "KLTN_prep_v2").exists():
    dataset_root = extract_p / "KLTN_prep_v2"
else:
    dataset_root = extract_p

# Đọc danh sách 48 classes chuẩn từ repo hoặc từ YAML trong zip
REF_YAML = "/content/klcn2026/configs/data/exp_2/data_70_15_15.yaml"
if os.path.exists(REF_YAML):
    with open(REF_YAML, "r", encoding="utf-8") as f:
        class_names = yaml.safe_load(f).get("names", [])
else:
    class_names = []

# Tìm file train/val/test txt hoặc images directory
train_candidates = list(dataset_root.glob(f"*train*_{SELECTED_RATIO}.txt")) or list(dataset_root.glob("*train*.txt"))
val_candidates = list(dataset_root.glob(f"*val*_{SELECTED_RATIO}.txt")) or list(dataset_root.glob("*val*.txt"))
test_candidates = list(dataset_root.glob(f"*test*_{SELECTED_RATIO}.txt")) or list(dataset_root.glob("*test*.txt"))

if train_candidates and val_candidates:
    train_path = str(train_candidates[0])
    val_path = str(val_candidates[0])
    test_path = str(test_candidates[0]) if test_candidates else val_path
elif (dataset_root / "dataset" / "images").exists():
    train_path = str(dataset_root / "dataset" / "images")
    val_path = train_path
    test_path = train_path
else:
    train_path = str(dataset_root)
    val_path = str(dataset_root)
    test_path = str(dataset_root)

colab_yaml_content = {
    "path": str(dataset_root),
    "train": train_path,
    "val": val_path,
    "test": test_path,
    "nc": len(class_names),
    "names": class_names
}

ACTIVE_YAML_PATH = "/content/klcn2026/configs/data/exp_2/data_colab_active.yaml"
os.makedirs(os.path.dirname(ACTIVE_YAML_PATH), exist_ok=True)
with open(ACTIVE_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(colab_yaml_content, f, allow_unicode=True, sort_keys=False)

print(f"✅ Đã tạo file YAML cấu hình chuẩn tại: {ACTIVE_YAML_PATH}")
print("\n--- Nội dung YAML ---")
!cat {ACTIVE_YAML_PATH}

## 🚀 Bước 5️⃣: GIAI ĐOẠN 2 — Thực Nghiệm Huấn Luyện (Baseline & CBAM Variants)
> Cấu hình thư mục lưu kết quả trên Google Drive để không bị mất khi hết phiên Colab.

In [ ]:
RUNS_DIR = f"/content/drive/MyDrive/KLTN-2026-testingNtraining/runs_exp2_{SELECTED_RATIO}"
os.makedirs(f"{RUNS_DIR}/baseline", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_backbone", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_neck", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_head", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_backbone_head", exist_ok=True)

print(f"✅ Thư mục lưu kết quả trên Drive: {RUNS_DIR}")

### Cell 5A: Huấn Luyện YOLO11n BASELINE (Exp 0)

In [ ]:
!python src/training/train_baseline.py \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/baseline" \
    --name train

### Cell 5B: Huấn Luyện CBAM BACKBONE (Exp 1)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_backbone" \
    --name train

### Cell 5C: Huấn Luyện CBAM NECK (Exp 2)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_neck.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_neck" \
    --name train

### Cell 5D: Huấn Luyện CBAM HEAD (Exp 4)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_head" \
    --name train

### Cell 5E: Huấn Luyện CBAM BACKBONE + HEAD (Biến Thể Phối Hợp)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_backbone_head" \
    --name train

### 🔄 Cell Tiếp Tục Khi Bị Crash (Resume)

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

RESUME_CHECKPOINT = f"{RUNS_DIR}/cbam_backbone/train/weights/last.pt"

if os.path.exists(RESUME_CHECKPOINT):
    print(f"🔄 Đang tiếp tục huấn luyện từ: {RESUME_CHECKPOINT}")
    model = YOLO(RESUME_CHECKPOINT)
    model.train(resume=True)
else:
    print(f"❌ Không tìm thấy checkpoint: {RESUME_CHECKPOINT}")